# Prototype Extraction

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Concept formation

Can the model extract prototypes (central tendencies) from category examples?
Pre/post learning framework: learn what makes a "typical" member of each category.

**Difficulty grid:** category overlap × num examples × 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

def extract_number(response: str) -> str:
    """Extract a number from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean): return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

def extract_category(response: str, categories: list) -> str:
    """Extract a category name from model response, given valid category names."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    # Check short lines from bottom up for exact category name
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?').strip()
        for cat in categories:
            if clean.lower() == cat.lower():
                return cat
        if len(clean) <= 30:
            for cat in categories:
                if cat.lower() in clean.lower():
                    return cat
    # Fallback: search entire text from bottom
    for line in reversed(lines):
        for cat in categories:
            if cat.lower() in line.lower():
                return cat
    return ''

In [ ]:
# Constants used in results analysis
OVERLAP_NOISE = {'low': 1, 'medium': 2, 'high': 3}
EVIDENCE = {'few': 4, 'mid': 8, 'many': 12}
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/prototype_extraction_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/prototype_extraction_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')

In [ ]:
PRE_P = """You are studying creatures on an alien planet. Scientists have classified them into categories.

Here are observed creatures and their categories:
{material}

A new creature has been found:
{test_input}

Which category does this creature belong to: Zorplings, Minkles, or Quibbles?

Reply with ONLY the category name."""

STUDY_P = """You are studying creatures on an alien planet. Scientists have classified them into categories.

Here are observed creatures and their categories:
{material}

Analyze these creatures carefully:
1. For each category (Zorplings, Minkles, Quibbles), compute the AVERAGE value of each feature across all examples.
2. Describe the PROTOTYPE (typical member) of each category.
3. Identify which features most distinguish each category from the others.
4. Verify your prototypes by checking that each example is closest to its own category's prototype.
5. If any don't match, revise your prototypes.

Show all work."""

POST_P = """You previously analyzed alien creature categories and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original observed creatures:
{material}

A new creature has been found:
{test_input}

Based on your analysis of the category prototypes, which category does this creature belong to: Zorplings, Minkles, or Quibbles?

Reply with ONLY the category name."""

## Evaluation

In [ ]:
all_results = []

available = list(kbench.llms.keys())
@kbench.task(name='prototype_extraction',
             description='Pre/post learning prototype-based creature classification')
def prototype_extraction(llm, material: str, test_input: str, expected: str,
                         overlap: str, evidence: str, difficulty_label: str,
                         seed: int, n_examples: int, noise: int,
                         categories: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{overlap}_{evidence}_{seed}'
    cats = json.loads(categories)

    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_category(pre_raw, cats)
    pre_correct = pre_extracted == expected

    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_category(post_raw, cats)
    post_correct = post_extracted == expected

    all_results.append({
        'timestamp': ts, 'model': model_name, 'task_id': tid,
        'overlap': overlap, 'evidence': evidence, 'difficulty_label': difficulty_label,
        'seed': int(seed), 'n_examples': int(n_examples), 'noise': int(noise),
        'test_input': test_input, 'expected': expected,
        'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
        'study_notes': notes,
        'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
        'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
    })

    b,l = 'Y' if pre_correct else 'N', 'Y' if post_correct else 'N'
    print(f'[{model_name:40s}] [{difficulty_label:18s}] "{test_input[:40]}..." -> {expected}  pre={pre_extracted}({b})  post={post_extracted}({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

try:
    runs = prototype_extraction.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                      n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

# Per-model summary
print('SCALING CHECK — Pre-learning accuracy by model:')
print('=' * 70)
for model in sorted(results['model'].unique()):
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# Per model x overlap
print()
print('By model x overlap (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for o in OVERLAP_NOISE:
        sub = results[(results['model'] == model) & (results['overlap'] == o)]
        if len(sub) > 0:
            row += f'  {o}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per model x evidence
print()
print('By model x evidence (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for e in EVIDENCE:
        sub = results[(results['model'] == model) & (results['evidence'] == e)]
        if len(sub) > 0:
            row += f'  {e}={sub["pre_correct"].mean():.0%}'
    print(row)

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Overlap: {row["overlap"]}, Evidence: {row["evidence"]}, Noise: {row["noise"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Prototype Extraction: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty (overlap x evidence)')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('prototype_results.png', dpi=150, bbox_inches='tight')
plt.show()